##02-2. 지하철 승하차

* API로 받은 **원본 그대로** PostgreSQL 테이블에 만들고 저장 (파일로 저장하지 않음)
* 아래 셀은 앞의 셀이 먼저 실행되어 있어야 함
  * `conn`, `ensure_conn()`, `save_df()` : DB 연결, DB 저장 공통함수
  * `fetch_all_rows()`, `check_api()` : 서울 열린데이터광장 API 수집 함수
  * `SEOUL_API_KEY`, `START_DATE`, `END_DATE` : 인증키, 수집 기간
* 컬럼은 API 응답 컬럼명을 소문자로 바꿔 전부 TEXT로 저장
* 추정값은 원본 테이블에 섞지 않고 이후 별도 테이블로 만듦

###02-2-0. 공통 설정과 수집·저장 함수

데이터셋은 `RAW_SOURCES`에 한 줄씩 등록하고, 수집·저장은 하나의 함수(`load_source`)로 처리

| 이름 | 데이터셋 | 조건값 | 상태 |
|---|---|---|---|
| `card_time_monthly` | 서울시 월별·시간대별 승하차 (OA-12252, `CardSubwayTime`) | 월 `YYYYMM` | 확인됨 |
| `daily_total` | 서울시 지하철호선별 역별 승하차 인원 (OA-12914, 일별) | 일 `YYYYMMDD` | 서비스명 **미확인 후보** → 아래 확인 셀로 검증 |
| `metro_daily_hour` | 서울교통공사 일별·시간대별 승하차 (OA-12921) | 일 | OpenAPI가 아닌 **파일(CSV) 제공**으로 보임 → 서비스명이 없으면 실행 전에 멈춤 |

* `service` 확인 방법: 데이터셋 페이지 **OpenAPI 탭의 샘플url** `http://openapi.seoul.go.kr:8088/sample/json/서비스명/1/5/조건값`
* `freq`: `"D"` 일(YYYYMMDD) / `"M"` 월(YYYYMM) / `""` 조건값 없음(1회 전체 조회)

In [ ]:
# ============================================================
# [공통 1] 설정값 : 데이터셋별 서비스명 / 저장 테이블 / 조건값 단위
#   - service 가 비어 있으면 API 호출 전에 멈춤
# ============================================================
RAW_SOURCES = {
    "card_time_monthly": {                     # 서울시 월별·시간대별 승하차 (OA-12252)
        "service": "CardSubwayTime",
        "table":   "RAW_SUBWAY_CARD_TIME",
        "freq":    "M",
    },
    "daily_total": {                           # 서울시 지하철호선별 역별 승하차 인원, 일별 (OA-12914)
        "service": "CardSubwayStatsNew",       # ※ 미확인 후보: [일별 2] 확인 셀에서 검증할 것
        "table":   "RAW_SUBWAY_DAILY_TOTAL",
        "freq":    "D",
    },
    "metro_daily_hour": {                      # 서울교통공사 일별·시간대별 승하차 (OA-12921)
        "service": "",                         # ← OpenAPI 서비스명이 있을 때만 입력
        "table":   "RAW_SUBWAY_METRO_DAILY_HOUR",
        "freq":    "D",
    },
}
LOAD_LOG     = "SUBWAY_RAW_LOAD_LOG"           # 데이터셋 + 기간별 적재 이력 테이블
MAX_RETRIES  = 3                               # 통신 실패 시 재시도 횟수
PERIOD_FORMAT = {"D": "%Y%m%d", "M": "%Y%m", "": None}


def check_sources(names=None):
    """설정 확인: 서비스명이 비어 있거나 freq가 잘못되면 호출 전에 중단"""
    for name in (names or RAW_SOURCES):
        cfg = RAW_SOURCES[name]
        if not cfg["service"].strip():
            raise ValueError(
                f"[{name}] service가 비어 있습니다. 데이터셋 페이지 OpenAPI 탭의 "
                f"샘플url에서 서비스명을 복사해 RAW_SOURCES에 넣으세요.")
        if cfg["freq"] not in PERIOD_FORMAT:
            raise ValueError(f"[{name}] freq는 'D', 'M', '' 중 하나여야 합니다: {cfg['freq']!r}")


def check_source(name, date_text="2024-01-02"):
    """api 정상접속 확인 : 첫 5행만 조회해서 컬럼 / 기간 / 단위 확인"""
    check_sources([name])
    cfg = RAW_SOURCES[name]
    fmt = PERIOD_FORMAT[cfg["freq"]]
    param = pd.Timestamp(date_text).strftime(fmt) if fmt else ""
    return check_api(service=cfg["service"], date_parameter=param)


def table_exists(table_name):
    with conn.cursor() as cur:
        cur.execute("SELECT to_regclass(%s::text)", [table_name.lower()])
        exists = cur.fetchone()[0] is not None
    conn.rollback()                                 # 확인용 트랜잭션 정리
    return exists

In [ ]:
# ============================================================
# [공통 2] 기간별 수집 → 저장 함수
#   - 한 기간(하루/한 달) 수집이 끝난 뒤에만 저장, 데이터 + 이력을 한 트랜잭션으로 커밋
#   - 통신/JSON 응답 실패는 최대 3회 재시도, 인증키/API 오류는 그대로 중단
#   - 이력 status : DONE(저장 완료) / EMPTY(빈 응답, 저장 안 함)
# ============================================================
def ensure_load_log():
    ensure_conn()
    with conn.cursor() as cur:
        cur.execute(f"""CREATE TABLE IF NOT EXISTS {LOAD_LOG} (
            dataset   TEXT NOT NULL,
            period    TEXT NOT NULL,
            row_cnt   BIGINT,
            status    TEXT,
            loaded_at TIMESTAMP DEFAULT now(),
            PRIMARY KEY (dataset, period))""")
    conn.commit()


def get_load_log(dataset):
    """{기간: 상태}"""
    with conn.cursor() as cur:
        cur.execute(f"SELECT period, status FROM {LOAD_LOG} WHERE dataset = %s", [dataset])
        log = {r[0]: r[1] for r in cur.fetchall()}
    conn.rollback()
    return log


def set_load_log(dataset, period, row_cnt, status, commit=True):
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {LOAD_LOG} (dataset, period, row_cnt, status) VALUES (%s, %s, %s, %s)
            ON CONFLICT (dataset, period) DO UPDATE
            SET row_cnt = EXCLUDED.row_cnt, status = EXCLUDED.status, loaded_at = now()""",
            [dataset, period, row_cnt, status])
    if commit:
        conn.commit()


def fetch_period(service, param):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fetch_all_rows(service, param)
        except RuntimeError as error:
            if "행 요청 실패" not in str(error):
                raise
            print(f"[요청 실패] {attempt}/{MAX_RETRIES}회")
            if attempt == MAX_RETRIES:
                raise
            wait_seconds = attempt * 3
            print(f"{wait_seconds}초 후 {param or '전체'}를 다시 조회합니다.")
            time.sleep(wait_seconds)


def load_period(name, period, param):
    cfg = RAW_SOURCES[name]
    df_period = fetch_period(cfg["service"], param)

    ensure_conn()                                       # 수집하는 동안 끊겼으면 재접속
    try:
        if df_period.empty:
            # 빈 응답을 '승하차 0명'으로 확정하지 않고 기록만 남김
            set_load_log(name, period, 0, "EMPTY", commit=False)
        else:
            save_df(df_period, cfg["table"], conn, mode="INSERT", commit=False)
            set_load_log(name, period, len(df_period), "DONE", commit=False)
        conn.commit()                                   # 데이터 + 이력 동시 커밋
    except Exception:
        conn.rollback()
        raise
    return len(df_period)

In [ ]:
# ============================================================
# [공통 3] 기간 전체 반복 적재
#   - 완료(DONE) / 빈 응답(EMPTY)인 기간은 건너뜀 → 다시 실행하면 이어서 진행
#   - retry_empty=True 로 빈 응답이었던 기간만 다시 조회 가능
#   - max_periods 로 일부만 먼저 시험 가능 (예: 3)
# ============================================================
def load_source(name, max_periods=None, retry_empty=False, stop_on_error=True):
    check_sources([name])
    cfg = RAW_SOURCES[name]
    fmt = PERIOD_FORMAT[cfg["freq"]]
    ensure_conn()
    ensure_load_log()

    if fmt is None:                                     # 조건값 없는 데이터셋: 1회 전체 조회
        periods = [("ALL", "")]
    else:
        periods = [(p.strftime(fmt), p.strftime(fmt))
                   for p in pd.period_range(START_DATE, END_DATE, freq=cfg["freq"])]

    log = get_load_log(name)
    skip = {"DONE"} if retry_empty else {"DONE", "EMPTY"}
    todo = [(p, q) for p, q in periods if log.get(p) not in skip]
    if max_periods:
        todo = todo[:max_periods]

    n_done = sum(1 for p, _ in periods if log.get(p) == "DONE")
    print(f"[{name}] 전체 {len(periods)}개 / 완료 {n_done}개 / 이번 처리 {len(todo)}개")

    failed = {}
    for i, (period, param) in enumerate(todo, 1):       # ← 기간별 반복문
        print(f"\n[{name} {i}/{len(todo)}] {period}")
        t0 = time.time()
        try:
            n = load_period(name, period, param)
            print(f"   ✔ {n:,}행 / {time.time()-t0:,.0f}초")
        except Exception as e:
            failed[period] = str(e)
            print(f"   ⚠ {period} 실패: {e}")
            if stop_on_error:
                print("   → 중단합니다. 원인 해결 후 다시 실행하면 이 기간부터 이어서 진행합니다.")
                break
        time.sleep(0.2)

    print(f"\n[{name}] 처리 종료: 실패 {len(failed)}개", failed if failed else "")
    return failed

###02-2-1. 지하철 역 정보

서울시 역사마스터 정보 (`subwayStationMaster`) > 출처: 서울 열린데이터광장

* 역ID·역명·호선·위도·경도를 확인할 수 있는 원본 테이블 `RAW_SUBWAY_STATION_MASTER` 생성
* 환승역 묶기, 좌표 검증은 저장 이후 단계에서 진행

In [ ]:
# api 정상접속 확인 : 첫 5행만 조회
df_subway_master_check = check_api(service="subwayStationMaster")

In [ ]:
# ============================================================
# [역 정보 DB 저장] API 전체 수집 → RAW_SUBWAY_STATION_MASTER 생성 + 저장
# ============================================================
MASTER_TABLE = "RAW_SUBWAY_STATION_MASTER"

if table_exists(MASTER_TABLE):
    print(f"{MASTER_TABLE}: 이미 있는 테이블이라 저장하지 않습니다.")
    print("다시 넣으려면 DROP TABLE 후 이 셀을 다시 실행하세요.")
else:
    raw_subway_master = fetch_all_rows("subwayStationMaster")   # 1,000행씩 끝까지 수집

    print("역사마스터:", raw_subway_master.shape)
    print("원본 컬럼:", raw_subway_master.columns.tolist())
    display(raw_subway_master.head())

    if raw_subway_master.empty:
        raise RuntimeError("수집된 역사마스터 데이터가 없어 저장하지 않습니다.")

    save_df(raw_subway_master, MASTER_TABLE, conn)

###02-2-2. 월별 승하차 정보

교통카드 정산시스템 시간대별 승하차 (`CardSubwayTime`, OA-12252) > 출처: 서울 열린데이터광장

* 조건값은 사용월(YYYYMM). 월별로 API 호출 → `RAW_SUBWAY_CARD_TIME`에 저장
* 월별 진행 상황은 `SUBWAY_RAW_LOAD_LOG`에 기록 (`DONE` 완료 / `EMPTY` 빈 응답)

In [ ]:
# api 정상접속 확인 : 2021년 1월 첫 5행만 조회
df_subway_time_check = check_source("card_time_monthly", "2021-01-01")

In [ ]:
# 먼저 1개월만 시험 저장해서 테이블이 잘 만들어지는지 확인
failed = load_source("card_time_monthly", max_periods=1)

In [ ]:
# 전체 기간 저장
failed = load_source("card_time_monthly")

In [ ]:
# ============================================================
# [이전 버전 이력 이전] 이전에 SUBWAY_CARD_TIME_LOAD_LOG 로 월별 적재를 이미 실행했다면
#   그 기록을 새 이력 테이블로 옮겨서 이미 받은 달을 다시 받지 않도록 함 (없으면 건너뜀)
# ============================================================
OLD_LOG = "SUBWAY_CARD_TIME_LOAD_LOG"
if table_exists(OLD_LOG):
    ensure_load_log()
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {LOAD_LOG} (dataset, period, row_cnt, status, loaded_at)
            SELECT 'card_time_monthly', use_month, row_cnt, status, loaded_at FROM {OLD_LOG}
            ON CONFLICT (dataset, period) DO NOTHING""")
        print("이전 이력 이전:", cur.rowcount, "건")
    conn.commit()
else:
    print("이전 버전 이력 테이블이 없어 건너뜁니다.")

###02-2-3. 일별 승하차 정보

월별 자료만으로는 일별 분석이 안 되어 일별 자료를 원본 그대로 추가 저장

1. 서울시 지하철호선별 역별 승하차 인원, 일별 (OA-12914) > 추정값 보정용 실제 일별 총량 (서울교통공사·코레일·공항철도·9호선)
2. 서울교통공사 일별·시간대별 승하차 (OA-12921) > 날짜·요일에 따른 시간대별 이용 패턴 파악용
   * 파일(CSV)로만 제공되는 경우 API 대신 파일을 읽어 같은 방식으로 DB에 저장해야 함 (서비스명이 비어 있으면 실행되지 않음)

In [ ]:
# api 정상접속 확인 : 첫 5행으로 컬럼 / 기간 / 역·노선 단위 확인
# 서비스명이 틀렸다면 여기서 ERROR 가 나므로 전체 수집 전에 알 수 있음
df_total_check = check_source("daily_total")

In [ ]:
# 먼저 3개 기간만 시험 저장해서 테이블이 잘 만들어지는지 확인
failed = load_source("daily_total", max_periods=3)

In [ ]:
# 전체 기간 저장 (약 2,000일: 끊겨도 다시 실행하면 이어서 진행)
failed = load_source("daily_total")

In [ ]:
# 서울교통공사 일별·시간대별 (OpenAPI 서비스명이 RAW_SOURCES에 입력된 경우에만 실행)
df_metro_check = check_source("metro_daily_hour")
failed = load_source("metro_daily_hour", max_periods=3)

###02-2-4. 저장 결과 확인

In [ ]:
# ============================================================
# 저장 결과 확인 : 기간, 건수, 컬럼, 역/노선 단위 파악
# ============================================================
conn.rollback()      # 오류로 중단된 트랜잭션이 남아 있으면 정리 (필수)

for name, cfg in RAW_SOURCES.items():
    table = cfg["table"].lower()
    print(f"\n===== {name} ({table}) =====")
    if not table_exists(cfg["table"]):
        print("테이블이 아직 없습니다.")
        continue
    with conn.cursor() as cur:
        cur.execute(f"SELECT status, COUNT(*), SUM(row_cnt) FROM {LOAD_LOG} "
                    "WHERE dataset = %s GROUP BY status ORDER BY status", [name])
        for status, n_periods, n_rows in cur.fetchall():
            print(f"{status}: {n_periods}개 기간 / {int(n_rows or 0):,}행")

        cur.execute(f"SELECT COALESCE(SUM(row_cnt), 0) FROM {LOAD_LOG} "
                    "WHERE dataset = %s AND status = 'DONE'", [name])
        log_rows = int(cur.fetchone()[0])
        cur.execute(f"SELECT COUNT(*) FROM {table}")
        real_rows = cur.fetchone()[0]
        print(f"실제 행 수: {real_rows:,}  →  로그와 {'일치 ✔' if real_rows == log_rows else '불일치 ⚠'}")

        cur.execute(f"SELECT * FROM {table} LIMIT 3")
        print("컬럼:", [d[0] for d in cur.description])
        for r in cur.fetchall():
            print(" ", r)
    conn.rollback()

##02-3. 지하철역 - 따릉이 대여소 근접 관계

* **지도 API는 사용하지 않음**: 지하철역과 대여소 모두 DB(원본 테이블)에 위도·경도가 있으므로 좌표로 직접 계산
  * 지도 API가 필요한 경우: 좌표가 없는 대여소를 주소로 변환(지오코딩)하거나, 도보 경로 거리가 필요할 때
* 근처 기준: 두 지점의 **직선거리(haversine)**
  * 300m 이내 : 가까움 / 300~500m : 보통 / 500m 초과 : 연결하지 않음
* 순서: ① 원본 컬럼 확인 → ② 컬럼 지정 → ③ DataFrame 생성·좌표 검증 → ④ 거리 계산 → ⑤ 확인 → ⑥ 테이블 생성
* 결과 테이블 `STATION_SUBWAY_LINK` : 반경 안의 (대여소, 지하철역) 쌍, 가까운 순서와 가장 가까운 역 여부 포함

In [ ]:
# ============================================================
# [근접 1] 설정값 + 공통 함수
#   - 원본 테이블에 맞게 이름만 바꿔 쓰면 됨
#   - 대여소 정보(반기)가 아닌 대여소 마스터를 쓰려면 BIKE_RAW_TABLE 을 바꿀 것
# ============================================================
import re
import numpy as np

SUBWAY_RAW_TABLE = "raw_subway_station_master"      # 02-2-1 에서 저장한 역사마스터
BIKE_RAW_TABLE   = "raw_bike_station"               # 02-1-1 에서 저장한 대여소 정보 (또는 raw_bike_station_master)
LINK_TABLE       = "STATION_SUBWAY_LINK"            # 새로 만들 결과 테이블

NEAR_RADIUS_M     = 300     # 가까움 구간 경계
MAX_LINK_RADIUS_M = 500     # 연결 최대 반경
SAME_STATION_M    = 700     # 같은 이름의 역이 이 거리 이상 벌어지면 다른 역으로 분리

# 역명 표기가 다를 때 수동으로 맞추는 표 예) {"총신대입구(이수)": "이수"}
MANUAL_NAME_MAP = {}


def read_df(query, params=None):
    """SELECT 결과를 DataFrame 으로 (원본 테이블은 전부 TEXT)"""
    ensure_conn()
    with conn.cursor() as cur:
        cur.execute(query, params)
        df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    conn.rollback()
    return df


def pick_col(columns, candidates, label, required=True):
    """후보 이름 중 실제 존재하는 컬럼명 반환 (대소문자 무시)"""
    lookup = {str(c).strip().upper(): c for c in columns}
    for cand in candidates:
        if cand.upper() in lookup:
            return lookup[cand.upper()]
    if required:
        raise ValueError(f"[{label}] 컬럼을 찾지 못했습니다. 후보: {candidates}\n  현재 컬럼: {list(columns)}")
    return None


def haversine_m(lat1, lon1, lat2, lon2):
    """두 좌표 사이 직선거리(m). numpy 배열 브로드캐스팅 지원"""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 6371008.8 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def normalize_name(name):
    """역명 정규화: 수동 매핑 → 괄호 내용 제거 → 공백 제거 → 끝의 '역' 제거"""
    s = str(name).strip()
    s = MANUAL_NAME_MAP.get(s, s)
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"\s+", "", s)
    return re.sub(r"역$", "", s)


def normalize_line(line):
    """호선 정규화: '01호선', '1호선', '1' → '1호선' / 그 외 노선명은 공백만 제거"""
    s = str(line).strip()
    m = re.fullmatch(r"0*(\d+)\s*(호선)?", s)
    return f"{int(m.group(1))}호선" if m else re.sub(r"\s+", "", s)


def fix_coords(df, lat_col, lon_col):
    """좌표를 숫자로 바꾸고, 위경도가 뒤바뀐 행은 교정, 범위 밖/결측 행은 오류 표로 분리"""
    out = df.copy()
    out[lat_col] = pd.to_numeric(out[lat_col], errors="coerce")
    out[lon_col] = pd.to_numeric(out[lon_col], errors="coerce")
    swapped = out[lat_col].between(124, 132) & out[lon_col].between(33, 39)
    out.loc[swapped, [lat_col, lon_col]] = out.loc[swapped, [lon_col, lat_col]].to_numpy()
    valid = out[lat_col].between(37.0, 38.2) & out[lon_col].between(126.0, 128.0)
    return out[valid].copy(), out[~valid].copy(), int(swapped.sum())

In [ ]:
# ============================================================
# [근접 2] 원본 컬럼 확인 : 좌표 컬럼이 어떤 이름으로 들어와 있는지 먼저 확인
# ============================================================
for table in (SUBWAY_RAW_TABLE, BIKE_RAW_TABLE):
    if not table_exists(table):
        raise RuntimeError(f"{table} 테이블이 없습니다. 앞 단계 저장 여부를 확인하세요.")
    df_peek = read_df(f"SELECT * FROM {table} LIMIT 3")
    n_rows = read_df(f"SELECT COUNT(*) AS n FROM {table}")["n"].iloc[0]
    print(f"\n===== {table} ({n_rows:,}행) =====")
    print("컬럼:", df_peek.columns.tolist())
    display(df_peek)

In [ ]:
# ============================================================
# [근접 3] 사용할 컬럼 지정 : 후보 이름에서 자동 탐지, 틀리면 직접 덮어쓰기
#   - 아래 출력된 매핑이 맞는지 반드시 확인 (id/name 은 rental_history 와 이어질 값으로 선택)
# ============================================================
sub_cols  = read_df(f"SELECT * FROM {SUBWAY_RAW_TABLE} LIMIT 0").columns
bike_cols = read_df(f"SELECT * FROM {BIKE_RAW_TABLE} LIMIT 0").columns

SUBWAY_COLS = {
    "id":   pick_col(sub_cols, ["BLDN_ID", "STATN_ID", "STATION_ID", "SBWY_STNS_ID"], "역ID"),
    "name": pick_col(sub_cols, ["BLDN_NM", "STATN_NM", "STATION_NM", "SBWY_STNS_NM"], "역명"),
    "line": pick_col(sub_cols, ["ROUTE", "LINE_NUM", "LINE_NM", "SBWY_ROUT_LN_NM"], "호선"),
    "lat":  pick_col(sub_cols, ["LAT", "LATITUDE"], "위도"),
    "lon":  pick_col(sub_cols, ["LOT", "LNG", "LON", "LONGITUDE"], "경도"),
}
BIKE_COLS = {
    "id":   pick_col(bike_cols, ["RENT_ID", "STATION_ID", "STATN_ID", "STA_ID", "RENT_NO"], "대여소ID"),
    "name": pick_col(bike_cols, ["RENT_NM", "STATION_NAME", "STATN_NM", "RENT_ID_NM", "STA_NM"], "대여소명"),
    "lat":  pick_col(bike_cols, ["STA_LAT", "LATITUDE", "LAT", "STATN_LAT"], "위도"),
    "lon":  pick_col(bike_cols, ["STA_LONG", "STA_LNG", "LONGITUDE", "LON", "LNG", "STATN_LNT", "STATN_LOT"], "경도"),
}
# 자동 탐지가 틀렸다면 아래처럼 직접 지정 → 예) BIKE_COLS["id"] = "rent_id"

print("지하철 컬럼 매핑:", SUBWAY_COLS)
print("대여소 컬럼 매핑:", BIKE_COLS)

In [ ]:
# ============================================================
# [근접 4] DataFrame 생성 + 좌표 검증
#   A. subway_points   : 호선별 역 1행
#   B. subway_stations : 환승역을 하나로 묶은 물리역 (위치 1곳 1행)
#   C. bike_stations   : 좌표가 유효한 대여소
# ============================================================
# ---------- A. 호선별 역 ----------
raw_sub = read_df(f"SELECT * FROM {SUBWAY_RAW_TABLE}")
sub_ok, sub_err, sub_swapped = fix_coords(raw_sub, SUBWAY_COLS["lat"], SUBWAY_COLS["lon"])

subway_points = pd.DataFrame({
    "subway_id":   sub_ok[SUBWAY_COLS["id"]].astype("string").str.strip(),
    "subway_name": sub_ok[SUBWAY_COLS["name"]].astype("string").str.strip(),
    "line":        sub_ok[SUBWAY_COLS["line"]].astype("string").str.strip(),
    "lat":         sub_ok[SUBWAY_COLS["lat"]],
    "lon":         sub_ok[SUBWAY_COLS["lon"]],
}).drop_duplicates(subset=["subway_id"]).reset_index(drop=True)
subway_points["name_key"] = subway_points["subway_name"].map(normalize_name)
subway_points["line_key"] = subway_points["line"].map(normalize_line)
print(f"[역] 원본 {len(raw_sub):,}행 → 유효 {len(subway_points):,}행 / 좌표 오류 {len(sub_err)}행 / 위경도 뒤바뀜 교정 {sub_swapped}행")

# ---------- B. 물리역 (같은 역명 = 같은 위치, 너무 멀면 호선별로 분리) ----------
subway_points["phys_key"] = subway_points["name_key"]
for name_key, g in subway_points.groupby("name_key"):
    spread = haversine_m(g["lat"], g["lon"], g["lat"].mean(), g["lon"].mean()).max()
    if spread > SAME_STATION_M:
        subway_points.loc[g.index, "phys_key"] = name_key + "_" + g["line_key"]

subway_stations = (
    subway_points.groupby("phys_key")
    .agg(subway_name=("subway_name", "first"),
         lat=("lat", "mean"), lon=("lon", "mean"),
         lines=("line_key", lambda s: ", ".join(sorted(s.unique()))),
         n_lines=("line_key", "nunique"))
    .reset_index()
)
print(f"[물리역] {len(subway_stations):,}곳 / 환승역(2개 호선 이상) {(subway_stations['n_lines'] >= 2).sum():,}곳")

# ---------- C. 대여소 ----------
raw_bike = read_df(f"SELECT * FROM {BIKE_RAW_TABLE}")
bike_ok, bike_err, bike_swapped = fix_coords(raw_bike, BIKE_COLS["lat"], BIKE_COLS["lon"])
bike_stations = pd.DataFrame({
    "station_id":   bike_ok[BIKE_COLS["id"]].astype("string").str.strip(),
    "station_name": bike_ok[BIKE_COLS["name"]].astype("string").str.strip(),
    "latitude":     bike_ok[BIKE_COLS["lat"]],
    "longitude":    bike_ok[BIKE_COLS["lon"]],
})
n_dup = int(bike_stations.duplicated("station_id").sum())
bike_stations = bike_stations.drop_duplicates("station_id").reset_index(drop=True)
print(f"[대여소] 원본 {len(raw_bike):,}행 → 유효 {len(bike_stations):,}행 / 좌표 오류 {len(bike_err)}행 / "
      f"위경도 뒤바뀜 교정 {bike_swapped}행 / ID 중복 제거 {n_dup}행")

if len(sub_err):
    print("\n좌표 오류 역:"); display(sub_err.head(20))
if len(bike_err):
    print("\n좌표 오류 대여소:"); display(bike_err.head(20))

In [ ]:
# ============================================================
# [근접 5] 거리 계산 → 근접 관계 DataFrame (station_subway_link)
#   - 대여소 × 물리역 직선거리 행렬(numpy) → 반경 500m 안의 쌍만 남김
# ============================================================
dist = haversine_m(
    bike_stations["latitude"].to_numpy()[:, None],
    bike_stations["longitude"].to_numpy()[:, None],
    subway_stations["lat"].to_numpy()[None, :],
    subway_stations["lon"].to_numpy()[None, :],
)                                                   # shape = (대여소 수, 물리역 수)

bi, si = np.where(dist <= MAX_LINK_RADIUS_M)
station_subway_link = pd.DataFrame({
    "station_id":   bike_stations["station_id"].to_numpy()[bi],
    "station_name": bike_stations["station_name"].to_numpy()[bi],
    "phys_key":     subway_stations["phys_key"].to_numpy()[si],
    "subway_name":  subway_stations["subway_name"].to_numpy()[si],
    "lines":        subway_stations["lines"].to_numpy()[si],
    "distance_m":   dist[bi, si].round(1),
})
station_subway_link["radius_band"] = np.where(
    station_subway_link["distance_m"] <= NEAR_RADIUS_M,
    f"0-{NEAR_RADIUS_M}m", f"{NEAR_RADIUS_M}-{MAX_LINK_RADIUS_M}m")
# 한 대여소가 여러 역 반경에 걸칠 때 가까운 순서
station_subway_link["rank_in_station"] = (
    station_subway_link.groupby("station_id")["distance_m"].rank(method="first").astype(int))
station_subway_link["is_nearest"] = station_subway_link["rank_in_station"].eq(1)
station_subway_link["radius_m"] = MAX_LINK_RADIUS_M
station_subway_link = station_subway_link.sort_values(["station_id", "rank_in_station"]).reset_index(drop=True)

# 반경 밖 대여소도 포함한 '가장 가까운 역까지 거리' (확인용)
nearest_m = pd.Series(dist.min(axis=1), index=bike_stations["station_id"])
print(f"연결 쌍: {len(station_subway_link):,}건 / 역 근처 대여소: {station_subway_link['station_id'].nunique():,}곳 "
      f"/ 근처 대여소가 있는 역: {station_subway_link['phys_key'].nunique():,}곳")
display(station_subway_link.head(10))

In [ ]:
# ============================================================
# [근접 6] 기준 확인 : 거리 분포, 반경 밖 대여소, 상식 점검
# ============================================================
n_bike = len(bike_stations)
for r in (NEAR_RADIUS_M, MAX_LINK_RADIUS_M):
    print(f"가장 가까운 역이 {r}m 이내인 대여소: {(nearest_m <= r).mean():.1%}")
print(f"반경 {MAX_LINK_RADIUS_M}m 안에 역이 없는 대여소: {(nearest_m > MAX_LINK_RADIUS_M).sum():,}곳 / {n_bike:,}곳")

print("\n가장 가까운 역까지 거리(m) 분포")
display(nearest_m.describe().round(0).to_frame("nearest_m"))
print("역당 근처 대여소 수 분포")
display(station_subway_link.groupby("phys_key").size().describe().round(1).to_frame("대여소 수"))

# 좌표가 한쪽만 틀어진 경우 모든 대여소가 '반경 밖'으로 나옴 → 중심 간 거리로 점검
gap_km = haversine_m(bike_stations["latitude"].mean(), bike_stations["longitude"].mean(),
                     subway_stations["lat"].mean(), subway_stations["lon"].mean()) / 1000
print(f"\n대여소 중심 ↔ 역 중심 거리: {gap_km:.1f} km  (서울 안이면 수 km 이내가 정상)")
if (nearest_m > MAX_LINK_RADIUS_M).mean() > 0.9 or gap_km > 10:
    print("⚠ 거의 모든 대여소가 반경 밖입니다. 좌표 컬럼 매핑(근접 3)을 다시 확인하세요.")

# 상식 점검 : 합정 근처 대여소와 거리
check_name = "합정"
print(f"\n'{check_name}' 근처 대여소 (가까운 순)")
display(station_subway_link[station_subway_link["subway_name"].str.contains(check_name, na=False)]
        .sort_values("distance_m").head(10))

In [ ]:
# ============================================================
# [근접 7] 테이블 생성 + 저장 : STATION_SUBWAY_LINK
#   - save_df 는 모든 컬럼을 TEXT 로 저장(원본 보관용)하므로 거리/순위 같은 계산 결과에는 맞지 않음
#     → 형식을 지정해 직접 CREATE 하고 COPY 로 저장
#   - 계산으로 언제든 다시 만들 수 있는 테이블이라 한 트랜잭션에서 비우고 다시 채움
# ============================================================
import io

link_cols = ["station_id", "phys_key", "station_name", "subway_name", "lines",
             "distance_m", "radius_band", "rank_in_station", "is_nearest", "radius_m"]

if station_subway_link.empty:
    raise RuntimeError("저장할 근접 관계가 없습니다. 근접 3~6을 먼저 확인하세요.")

ensure_conn()
t = LINK_TABLE.lower()
try:
    with conn.cursor() as cur:
        cur.execute(f"""CREATE TABLE IF NOT EXISTS {t} (
            station_id      TEXT             NOT NULL,   -- 따릉이 대여소
            phys_key        TEXT             NOT NULL,   -- 물리역 키 (환승역은 1개)
            station_name    TEXT,
            subway_name     TEXT,
            lines           TEXT,                        -- 예: '2호선, 6호선'
            distance_m      DOUBLE PRECISION NOT NULL,   -- 직선거리(m)
            radius_band     TEXT,                        -- '0-300m' / '300-500m'
            rank_in_station INTEGER,                     -- 그 대여소에서 가까운 역 순서
            is_nearest      BOOLEAN,                     -- 그 대여소의 가장 가까운 역인지
            radius_m        INTEGER,                     -- 연결에 사용한 최대 반경
            built_at        TIMESTAMPTZ DEFAULT now(),
            PRIMARY KEY (station_id, phys_key))""")
        cur.execute(f"CREATE INDEX IF NOT EXISTS idx_{t}_phys ON {t} (phys_key)")
        cur.execute(f"TRUNCATE {t}")                      # 다시 실행해도 중복 없음

        buf = io.StringIO()
        station_subway_link[link_cols].to_csv(buf, index=False, header=False)
        buf.seek(0)
        cur.copy_expert(f"COPY {t} ({', '.join(link_cols)}) FROM STDIN WITH (FORMAT csv)", buf)
    conn.commit()
except Exception:
    conn.rollback()
    raise
print(f"{t.upper()}: {len(station_subway_link):,}행 저장 완료")

In [ ]:
# ============================================================
# [근접 8] 저장 결과 확인
# ============================================================
t = LINK_TABLE.lower()
with conn.cursor() as cur:
    cur.execute(f"SELECT COUNT(*), COUNT(DISTINCT station_id), COUNT(DISTINCT phys_key), "
                f"SUM(is_nearest::int) FROM {t}")
    n, n_station, n_subway, n_nearest = cur.fetchone()
    print(f"행 {n:,} / 대여소 {n_station:,}곳 / 역 {n_subway:,}곳 / 가장 가까운 역 표시 {n_nearest:,}건")
    print("DataFrame 과 행 수:", "일치 ✔" if n == len(station_subway_link) else "불일치 ⚠")

    cur.execute(f"""SELECT radius_band, COUNT(*), ROUND(AVG(distance_m)::numeric, 1)
                    FROM {t} GROUP BY radius_band ORDER BY radius_band""")
    for band, cnt, avg in cur.fetchall():
        print(f"  {band}: {cnt:,}건 (평균 {avg}m)")
conn.rollback()